In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the image
data = {
    "Year": list(range(2002, 2027)),
    "Jan": [None, 0.57, 2.78, 1.27, -0.85, 2.85, 5.09, 0.31, 0.87, 1.67, 1.37, 2.89, 0.72, 3.84, -1.20, 1.02, 5.43, 0.62, 1.80, -1.64, 0.53, 1.21, -0.72, 0.45, 3.76],
    "Feb": [None, 1.77, 0.61, 2.06, 0.96, -0.90, 5.28, 1.82, 0.68, 2.00, 2.05, 0.32, 2.78, 3.69, -3.69, 2.05, -3.74, 2.02, -0.51, 6.34, 0.24, 0.45, -0.93, 0.49, 2.38],
    "Mar": [None, -0.74, -2.00, -0.45, 0.28, 1.18, -0.67, 5.31, 1.03, 0.31, -0.69, 0.46, -3.04, 1.91, -0.98, -1.18, -1.07, 1.78, -4.46, -3.60, 4.71, -4.73, 3.10, -2.76, -1.86],
    "Apr": [1.68, -2.09, -2.52, 2.86, 1.03, 5.05, -5.44, 1.14, -0.31, 2.80, 0.29, 0.61, -2.27, -3.91, -0.68, 1.29, 0.24, 1.38, 0.01, 3.12, 5.37, 2.01, 3.37, -2.44, 4.55],
    "May": [2.42, 3.66, -1.14, 2.99, -0.80, 5.62, 2.76, 2.52, -0.13, -2.26, 1.58, 1.59, 2.17, 4.11, -0.69, 4.37, 4.44, 2.14, 5.81, -0.70, -3.29, 1.87, -2.01, -1.35, None],
    "Jun": [4.51, 2.43, 0.62, 6.22, -2.13, 1.73, 2.18, -0.36, -1.92, -3.28, -2.61, -3.20, 0.28, -2.04, 4.68, -0.97, -1.22, 4.37, 2.88, -2.34, 1.45, -1.10, 1.59, 3.03, None],
    "Jul": [3.37, -1.77, -1.21, 0.92, -1.29, 0.08, -1.82, 1.33, -0.89, 0.76, 3.08, 2.74, -0.84, 1.96, 3.63, 1.30, 0.74, 1.42, 2.13, -0.29, -1.21, -0.43, -1.42, 2.82, None],
    "Aug": [1.04, 1.43, 1.18, 0.41, 0.11, -7.33, 1.21, 1.18, 4.81, -0.97, 1.64, 2.08, 2.00, -5.15, -0.33, 4.90, 2.53, 1.85, 0.76, -2.25, 3.55, 0.79, 0.19, 1.59, None],
    "Sep": [4.72, 3.91, -0.23, 4.54, -0.45, 4.26, -2.20, 4.17, -0.88, 1.23, -0.53, 1.63, 1.19, -1.56, 0.67, -0.70, 1.56, -5.41, 0.14, 0.93, 4.25, 1.59, 2.41, 4.05, None],
    "Oct": [-1.79, 1.18, 2.24, -4.72, 1.26, 4.99, -0.75, -0.99, 0.76, 1.10, -1.86, 0.57, -3.87, -0.43, 0.04, 3.82, -7.56, -2.64, 0.34, 0.34, 0.43, -0.13, -0.64, 5.26, None],
    "Nov": [-0.27, 0.64, 6.97, 4.02, 3.71, -0.55, 3.09, 3.04, -2.65, -2.62, 0.55, 2.63, 2.57, 2.41, -2.77, 0.17, -4.19, 0.22, 6.19, -3.46, -4.40, -4.06, 5.64, 0.74, None],
    "Dec": [0.01, 3.55, 3.03, 2.58, 5.47, -1.94, -0.78, 0.55, 4.38, 1.16, 0.03, 3.26, 1.97, 0.38, 0.10, -1.65, 2.02, 0.86, 8.77, -0.39, 2.46, 1.23, 2.83, 1.71, None]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
# Convert month abbreviations to numeric strings (e.g., 'Jan' -> '01')
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Drops non-existent months (Jan-Mar 2002 and May-Dec 2026)
)

# Convert return values from percentages to floating fractions (e.g., 1.68 -> 0.0168)
df_final["Return"] = df_final["Return"] / 100

print(df_final)

         Return
Period         
2002-04  0.0168
2002-05  0.0242
2002-06  0.0451
2002-07  0.0337
2002-08  0.0104
...         ...
2025-12  0.0171
2026-01  0.0376
2026-02  0.0238
2026-03 -0.0186
2026-04  0.0455

[289 rows x 1 columns]


In [4]:
df_final.to_csv('./hf_returns/brummer.csv')

In [5]:
import pandas as pd

# 1. Reconstruct the raw grid data from the image (ordered reverse-chronologically as shown)
data = {
    "Year": list(range(2026, 2010, -1)),
    "Jan": [0.77, 2.96, 1.05, 0.57, 2.46, -0.45, 0.00, 3.50, 0.74, 2.18, -0.48, -1.56, 1.89, 1.21, 3.81, None],
    "Feb": [-0.42, 1.06, -0.20, -0.43, 0.81, 2.74, 1.17, 0.12, -2.57, -0.37, -3.09, 1.22, 4.44, -4.95, -4.30, None],
    "Mar": [-4.30, -1.74, 1.75, 0.47, 0.69, -0.65, 3.62, 0.72, -0.66, 0.72, -1.24, 0.92, -0.33, 3.32, -0.32, 1.93],
    "Apr": [3.09, 0.83, 0.78, 0.86, 2.69, 0.96, 3.21, 3.55, 0.48, -0.90, 0.71, 2.12, -2.94, 1.64, -1.77, 1.29],
    "May": [None, 1.46, 1.28, 1.04, -2.08, -0.23, 3.86, -0.17, -0.69, 0.56, 0.63, 2.42, 2.27, 1.45, 0.65, 0.73],
    "Jun": [None, 2.32, 0.26, -1.02, 1.32, -0.11, 2.40, 0.01, 0.67, -1.28, -0.29, 0.59, 1.69, 0.77, 1.94, -0.45],
    "Jul": [None, 0.54, -0.25, 0.02, -0.33, -0.17, 4.52, 2.29, 1.56, -0.08, 0.71, 0.63, 0.27, 1.76, 4.84, 2.87],
    "Aug": [None, 0.72, 0.59, 0.71, 0.62, 0.77, 2.57, 0.27, 0.11, 2.71, 2.26, 0.09, 0.95, 1.05, 1.22, -3.17],
    "Sep": [None, 1.13, 0.01, 0.37, 1.69, 2.98, -0.06, -1.35, 1.43, 0.46, 1.33, -1.18, 5.41, 4.67, 1.28, -4.01],
    "Oct": [None, 2.36, 1.20, -0.02, -0.11, -0.47, 1.74, -0.39, -2.91, 0.90, -0.32, -0.37, -3.43, 3.65, 1.26, -1.53],
    "Nov": [None, 2.43, 4.24, -0.97, -1.28, 1.47, 3.74, 0.57, -5.59, -0.97, -1.07, 0.53, 1.47, 2.83, -0.14, -3.36],
    "Dec": [None, 1.08, 1.91, 1.06, 3.41, 0.81, 2.96, 2.23, 0.86, 2.88, 0.86, 0.94, 2.06, 2.43, 4.19, 0.79]
}

df_raw = pd.DataFrame(data)

# 2. Flatten the matrix structure into a single sequence
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Standardize month representations
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a proper time-series PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final cleaning and formatting
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()     # Put the index back into forward chronological order (2011 to 2026)
    .dropna()         # Drops non-existent months (e.g., Jan/Feb 2011 and May-Dec 2026)
)

# Convert percentage integers/floats into raw decimal returns (e.g., 0.77 -> 0.0077)
df_final["Return"] = df_final["Return"] / 100

print(df_final)

         Return
Period         
2011-03  0.0193
2011-04  0.0129
2011-05  0.0073
2011-06 -0.0045
2011-07  0.0287
...         ...
2025-12  0.0108
2026-01  0.0077
2026-02 -0.0042
2026-03 -0.0430
2026-04  0.0309

[182 rows x 1 columns]


In [6]:
df_final.to_csv('./hf_returns/bam.csv')

In [7]:
import pandas as pd

# 1. Reconstruct the raw grid data from the image (ordered chronologically by year)
data = {
    "Year": [2023, 2024, 2025, 2026],
    "Jan": [1.83, 1.78, 3.08, 0.72],
    "Feb": [4.49, 1.48, 0.83, 0.42],
    "Mar": [5.07, 7.00, 0.78, 0.45],
    "Apr": [1.89, 1.30, 0.12, 0.32],
    "May": [-0.39, 1.85, 0.97, None],
    "Jun": [1.63, 0.70, 1.52, None],
    "Jul": [0.15, 1.26, 2.06, None],
    "Aug": [-0.38, 0.67, 1.57, None],
    "Sep": [-0.39, 0.82, 1.37, None],
    "Oct": [0.19, 1.13, 0.47, None],
    "Nov": [0.56, 3.96, 0.65, None],
    "Dec": [0.95, 4.00, 0.20, None]
}

df_raw = pd.DataFrame(data)

# 2. Flatten the wide matrix format into a single sequence
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Standardize month abbreviations to two-digit numeric strings
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a proper time-series PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final cleaning, sorting, and decimal conversion
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()     # Organizes chronologically from Jan 2023 onward
    .dropna()         # Drops the empty future months (May-Dec 2026)
)

# Convert percentage values to standard fractional returns (e.g., 1.83 -> 0.0183)
df_final["Return"] = df_final["Return"] / 100

print(df_final)

         Return
Period         
2023-01  0.0183
2023-02  0.0449
2023-03  0.0507
2023-04  0.0189
2023-05 -0.0039
2023-06  0.0163
2023-07  0.0015
2023-08 -0.0038
2023-09 -0.0039
2023-10  0.0019
2023-11  0.0056
2023-12  0.0095
2024-01  0.0178
2024-02  0.0148
2024-03  0.0700
2024-04  0.0130
2024-05  0.0185
2024-06  0.0070
2024-07  0.0126
2024-08  0.0067
2024-09  0.0082
2024-10  0.0113
2024-11  0.0396
2024-12  0.0400
2025-01  0.0308
2025-02  0.0083
2025-03  0.0078
2025-04  0.0012
2025-05  0.0097
2025-06  0.0152
2025-07  0.0206
2025-08  0.0157
2025-09  0.0137
2025-10  0.0047
2025-11  0.0065
2025-12  0.0020
2026-01  0.0072
2026-02  0.0042
2026-03  0.0045
2026-04  0.0032


In [8]:
df_final.to_csv('./hf_returns/m1.csv')

In [16]:
import pandas as pd

# 1. Reconstruct the raw data directly from the image
data = {
    "Date": [
        "4/30/2026", "3/31/2026", "2/27/2026", "1/30/2026", "12/31/2025", "11/30/2025", 
        "10/31/2025", "9/30/2025", "8/29/2025", "7/31/2025", "6/30/2025", "5/30/2025", 
        "4/30/2025", "3/31/2025", "2/28/2025", "1/31/2025", "12/31/2024", "11/30/2024", 
        "10/31/2024", "9/30/2024", "8/30/2024", "7/31/2024", "6/30/2024", "5/31/2024", 
        "4/30/2024", "3/29/2024", "2/29/2024", "1/31/2024", "12/29/2023", "11/30/2023", 
        "10/31/2023", "9/29/2023", "8/31/2023", "7/31/2023", "6/30/2023", "5/31/2023", 
        "4/30/2023", "3/31/2023", "2/28/2023", "1/31/2023", "12/31/2022", "11/30/2022", 
        "10/31/2022", "9/30/2022", "8/31/2022", "7/29/2022", "6/30/2022", "5/31/2022", 
        "4/29/2022", "3/31/2022", "2/28/2022", "1/31/2022"
    ],
    "Net_Return_SMA": [
        None, None, None, 11.12, -2.63, -0.19, -4.79, 12.08, 1.49, 0.06, 2.05, -4.85, 
        4.46, 4.10, 3.55, -3.93, -6.00, 0.02, -6.65, 8.17, 0.63, 2.83, -2.02, -4.96, 
        5.31, 2.69, 12.06, 0.10, 5.41, 9.83, -0.70, 0.23, 0.41, -0.20, -0.03, 2.22, 
        -1.05, 6.03, -4.20, 4.95, -0.16, -1.42, -3.18, 1.40, 8.74, 4.69, 2.45, 5.24, 
        3.28, 12.87, 5.07, -2.51
    ],
    "Net_Return_Founder_Class": [
        -3.66, -3.47, -3.09, None, None, None, None, None, None, None, None, None, 
        None, None, None, None, None, None, None, None, None, None, None, None, 
        None, None, None, None, None, None, None, None, None, None, None, None, 
        None, None, None, None, None, None, None, None, None, None, None, None, 
        None, None, None, None
    ]
}

# 2. Initialize DataFrame
df = pd.DataFrame(data)

# 3. Convert the Date strings into a clean pandas Datetime Index
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date")

# 4. Sort chronologically (oldest date 2022-01-31 to newest 2026-04-30)
df = df.sort_index()

# 5. Convert return values from raw percentages to decimal fractions (e.g., 11.12 -> 0.1112)
df["Net_Return_SMA"] = df["Net_Return_SMA"] / 100
df["Net_Return_Founder_Class"] = df["Net_Return_Founder_Class"] / 100

df_concat = df.sum(axis=1)

df_concat

Date
2022-01-31   -0.0251
2022-02-28    0.0507
2022-03-31    0.1287
2022-04-29    0.0328
2022-05-31    0.0524
2022-06-30    0.0245
2022-07-29    0.0469
2022-08-31    0.0874
2022-09-30    0.0140
2022-10-31   -0.0318
2022-11-30   -0.0142
2022-12-31   -0.0016
2023-01-31    0.0495
2023-02-28   -0.0420
2023-03-31    0.0603
2023-04-30   -0.0105
2023-05-31    0.0222
2023-06-30   -0.0003
2023-07-31   -0.0020
2023-08-31    0.0041
2023-09-29    0.0023
2023-10-31   -0.0070
2023-11-30    0.0983
2023-12-29    0.0541
2024-01-31    0.0010
2024-02-29    0.1206
2024-03-29    0.0269
2024-04-30    0.0531
2024-05-31   -0.0496
2024-06-30   -0.0202
2024-07-31    0.0283
2024-08-30    0.0063
2024-09-30    0.0817
2024-10-31   -0.0665
2024-11-30    0.0002
2024-12-31   -0.0600
2025-01-31   -0.0393
2025-02-28    0.0355
2025-03-31    0.0410
2025-04-30    0.0446
2025-05-30   -0.0485
2025-06-30    0.0205
2025-07-31    0.0006
2025-08-29    0.0149
2025-09-30    0.1208
2025-10-31   -0.0479
2025-11-30   -0.0019
2025-12-

In [17]:
df_concat.to_csv('./hf_returns/arr.csv')

In [10]:
import pandas as pd

# 1. Reconstruct the fragmented tables into a unified chronological map
# Stripping out 'n/a' months (Jan-May 2019) and the summary 'Total' rows
data = {
    "Period": [
        # 2019
        "2019-06", "2019-07", "2019-08", "2019-09", "2019-10", "2019-11", "2019-12",
        # 2020
        "2020-01", "2020-02", "2020-03", "2020-04", "2020-05", "2020-06", 
        "2020-07", "2020-08", "2020-09", "2020-10", "2020-11", "2020-12",
        # 2021
        "2021-01", "2021-02", "2021-03", "2021-04", "2021-05", "2021-06", 
        "2021-07", "2021-08", "2021-09", "2021-10", "2021-11", "2021-12",
        # 2022
        "2022-01", "2022-02", "2022-03", "2022-04", "2022-05", "2022-06", 
        "2022-07", "2022-08", "2022-09", "2022-10", "2022-11", "2022-12",
        # 2023
        "2023-01", "2023-02", "2023-03", "2023-04", "2023-05", "2023-06", 
        "2023-07", "2023-08", "2023-09", "2023-10", "2023-11", "2023-12",
        # 2024
        "2024-01", "2024-02", "2024-03", "2024-04", "2024-05", "2024-06", 
        "2024-07", "2024-08", "2024-09", "2024-10", "2024-11", "2024-12",
        # 2025
        "2025-01", "2025-02", "2025-03", "2025-04", "2025-05", "2025-06", 
        "2025-07", "2025-08", "2025-09", "2025-10", "2025-11", "2025-12"
    ],
    "Return": [
        # 2019
        2.37, 2.61, 1.12, 2.81, 2.22, -0.36, 3.76,
        # 2020
        2.15, -2.06, -3.42, 1.96, 1.57, 1.49, 1.16, 1.47, 1.39, -0.08, -0.25, 0.70,
        # 2021
        1.89, -0.43, -1.40, 3.80, 1.40, 1.89, -0.38, 2.07, -0.37, -0.97, -1.51, -0.72,
        # 2022
        -2.58, 3.37, 1.44, 2.18, 0.11, 1.72, 1.66, 1.46, 0.23, -0.85, -1.17, 5.26,
        # 2023
        -1.73, -1.08, -2.53, -1.79, 0.13, 0.05, -0.17, 0.60, -1.25, -0.37, 0.04, 0.81,
        # 2024
        -0.21, 1.29, 2.50, 0.29, 2.37, 5.27, 2.16, 1.15, 0.12, 0.10, -0.25, 1.31,
        # 2025
        1.41, 4.49, 1.31, 1.23, 0.72, 1.38, 1.14, 3.32, -0.62, 2.67, -0.54, 1.56
    ]
}

# 2. Create the DataFrame
df = pd.DataFrame(data)

# 3. Establish a standard time-series PeriodIndex (YYYY-MM)
df["Period"] = pd.to_datetime(df["Period"]).dt.to_period("M")
df = df.set_index("Period")

# 4. Turn percentage integers into standard fractional returns (e.g., 2.37 -> 0.0237)
df["Return"] = df["Return"] / 100

print(df)

         Return
Period         
2019-06  0.0237
2019-07  0.0261
2019-08  0.0112
2019-09  0.0281
2019-10  0.0222
...         ...
2025-08  0.0332
2025-09 -0.0062
2025-10  0.0267
2025-11 -0.0054
2025-12  0.0156

[79 rows x 1 columns]


In [11]:
df.to_csv('./hf_returns/four_world.csv')

In [12]:
import pandas as pd

# 1. Reconstruct the raw grid data from the image (ordered reverse-chronologically)
data = {
    "Year": list(range(2026, 2016, -1)),
    "Jan": [4.44, 9.72, 3.75, -5.58, 0.77, -3.78, -0.33, 6.96, -1.47, 5.90],
    "Feb": [1.58, 4.01, 1.11, -3.15, 3.87, -5.87, 1.85, 3.48, 6.58, -1.04],
    "Mar": [5.24, 0.24, 5.37, -6.56, -4.44, -5.02, 2.62, 2.49, 1.71, 5.03],
    "Apr": [1.15, -3.35, 0.91, -7.88, 3.42, -9.14, -4.33, 0.37, 12.01, -3.07],
    "May": [None, -1.12, 2.42, 21.06, 7.43, -5.37, -0.39, 1.41, 12.99, 1.71],
    "Jun": [None, -7.70, 5.90, 11.98, 11.88, 3.94, -5.60, 0.75, 4.49, 2.96],
    "Jul": [None, -3.03, 2.09, 5.25, 12.44, 18.84, -5.43, -1.82, 3.20, -0.17],
    "Aug": [None, 2.39, 8.08, -7.52, 8.95, 6.10, 5.71, 4.29, -2.20, -0.17],
    "Sep": [None, 2.55, 4.87, 11.02, 7.76, 7.28, 3.17, -1.98, 2.99, 3.33],
    "Oct": [None, 5.06, 3.47, 2.95, 1.68, 9.55, 7.01, 3.32, 3.07, 9.79],
    "Nov": [None, 7.98, 2.41, 0.70, -1.99, 7.45, 9.84, 0.15, 4.87, -1.87],
    "Dec": [None, 3.84, 5.59, -1.47, -0.28, -0.17, 5.76, 6.85, 4.71, -0.17]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single sequential column
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Standardize month mapping
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a time-series PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Filter, sort, and normalize scale
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()     # Puts data into natural forward order (Jan 2017 -> Apr 2026)
    .dropna()         # Drops uncompleted months (May 2026 onwards)
)

# Convert return values from raw percentages to decimal fractions (e.g., 4.44 -> 0.0444)
df_final["Return"] = df_final["Return"] / 100

print(df_final)

         Return
Period         
2017-01  0.0590
2017-02 -0.0104
2017-03  0.0503
2017-04 -0.0307
2017-05  0.0171
...         ...
2025-12  0.0384
2026-01  0.0444
2026-02  0.0158
2026-03  0.0524
2026-04  0.0115

[112 rows x 1 columns]


In [13]:
df_final.to_csv('./hf_returns/vadantia.csv')

In [21]:
import pandas as pd

# Original data matrix from the screenshot
data = {
    2018: [None, None, None, None, None, None, None, None, None, None, 2.4, 2.8, 5.3],
    2019: [0.0, 11.0, 2.9, 27.3, 41.4, 0.8, -6.4, -5.1, -7.2, 4.5, -5.4, -3.8, 62.5],
    2020: [20.4, 0.7, -3.6, 11.2, 0.1, -3.3, 18.4, 5.0, -4.6, 12.3, 30.9, 22.3, 168.1],
    2021: [17.0, 23.9, -0.3, 5.5, 2.9, -10.4, 7.4, 16.3, -9.8, 18.0, -2.3, -9.7, 64.9],
    2022: [0.4, -11.0, 8.9, -6.6, 3.7, -3.2, 7.1, -5.2, -13.0, -1.0, -5.1, -3.2, -26.6],
    2023: [21.5, -7.7, 4.2, -3.3, -4.0, 1.5, -4.2, 0.4, -2.2, 14.3, 9.6, 10.5, 43.4],
    2024: [-1.2, 20.8, 8.4, -2.5, -0.6, -2.2, -0.2, -2.0, -0.3, -0.9, 18.3, -1.0, 38.8],
    2025: [-3.0, -3.5, -4.9, -0.1, 1.3, -2.2, 10.4, 3.0, -0.4, -4.3, -5.6, -5.6, -14.9],
    2026: [1.1, -0.4, -6.9, -2.4, None, None, None, None, None, None, None, None, -8.4]
}

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'YTD']
df = pd.DataFrame(data, index=months)

# 1. Exclude YTD row
df_months = df.loc[df.index != 'YTD']

# 2. Unstack matrix into long-form rows
df_long = df_months.unstack().reset_index()
df_long.columns = ['Year', 'Month', 'Return']

# 3. Map months to numerical values and build a Datetime format column
month_map = {m: i+1 for i, m in enumerate(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])}
df_long['Month_Num'] = df_long['Month'].map(month_map)
df_long['Date'] = pd.to_datetime(df_long['Year'].astype(str) + '-' + df_long['Month_Num'].astype(str) + '-01')

# Sort chronologically and keep relevant columns
df_ts = df_long.sort_values('Date').reset_index(drop=True)[['Date', 'Return']]

# 4. Trim leading and trailing empty periods
first_valid = df_ts['Return'].first_valid_index()
last_valid = df_ts['Return'].last_valid_index()
df_ts_clean = df_ts.loc[first_valid:last_valid].copy().reset_index(drop=True)

# Set the Date as index for time series features
df_ts_clean.set_index('Date', inplace=True)

df_ts_clean = df_ts_clean / 100
# Save the tidy format to CSV
# df_ts_clean.to_csv('monthly_returns_timeseries.csv')

print(df_ts_clean)

            Return
Date              
2018-11-01   0.024
2018-12-01   0.028
2019-01-01   0.000
2019-02-01   0.110
2019-03-01   0.029
...            ...
2025-12-01  -0.056
2026-01-01   0.011
2026-02-01  -0.004
2026-03-01  -0.069
2026-04-01  -0.024

[90 rows x 1 columns]


In [22]:
df_ts_clean.to_csv('./hf_returns/cambrian.csv')

In [23]:
import pandas as pd

# Extracting data from the new image
# Columns: JAN, FEB, MAR, APR, MAY, JUN, JUL, AUG, SEP, OCT, NOV, DEC, YEAR
data_new = {
    2021: [None, None, None, None, None, None, None, None, None, 1.2, 0.3, 0.9, 2.4],
    2022: [-1.6, -0.2, -1.7, 1.0, 0.6, -2.3, 2.4, -0.1, -0.8, -0.9, 0.8, 1.3, -1.5],
    2023: [1.0, -1.1, 0.3, 1.0, 0.2, 1.0, 2.3, 1.2, 1.8, 1.0, 3.0, 2.2, 14.8],
    2024: [-0.9, 3.3, 2.0, -1.5, 0.5, 3.8, 2.8, 0.9, 0.7, 1.0, 0.9, -1.8, 12.2],
    2025: [1.3, 2.5, -1.9, 0.8, 1.2, 1.4, 0.2, 1.7, 4.1, 2.6, -0.9, -1.6, 11.7],
    2026: [4.1, 1.3, 0.4, None, None, None, None, None, None, None, None, None, 5.8]
}

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'YEAR']
df_new = pd.DataFrame(data_new, index=months)

# 1. Exclude the annual 'YEAR' aggregate row
df_months = df_new.loc[df_new.index != 'YEAR']

# 2. Reshape/Unstack from matrix format to long format
df_long = df_months.unstack().reset_index()
df_long.columns = ['Year', 'Month', 'Return']

# 3. Create numerical month mappings and construct full Datetime index
month_map = {m: i+1 for i, m in enumerate(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])}
df_long['Month_Num'] = df_long['Month'].map(month_map)
df_long['Date'] = pd.to_datetime(df_long['Year'].astype(str) + '-' + df_long['Month_Num'].astype(str) + '-01')

# Sort chronologically and drop temporary columns
df_ts = df_long.sort_values('Date').reset_index(drop=True)[['Date', 'Return']]

# 4. Filter out leading and trailing missing data to isolate the true timeline
first_valid = df_ts['Return'].first_valid_index()
last_valid = df_ts['Return'].last_valid_index()
df_ts_clean = df_ts.loc[first_valid:last_valid].copy().reset_index(drop=True)

# Set Date as the formal index for your time-series analysis
df_ts_clean.set_index('Date', inplace=True)

df_ts_clean = df_ts_clean /100
# Save to a new CSV file
# df_ts_clean.to_csv('monthly_returns_timeseries_2.csv')

print(df_ts_clean)

            Return
Date              
2021-10-01   0.012
2021-11-01   0.003
2021-12-01   0.009
2022-01-01  -0.016
2022-02-01  -0.002
2022-03-01  -0.017
2022-04-01   0.010
2022-05-01   0.006
2022-06-01  -0.023
2022-07-01   0.024
2022-08-01  -0.001
2022-09-01  -0.008
2022-10-01  -0.009
2022-11-01   0.008
2022-12-01   0.013
2023-01-01   0.010
2023-02-01  -0.011
2023-03-01   0.003
2023-04-01   0.010
2023-05-01   0.002
2023-06-01   0.010
2023-07-01   0.023
2023-08-01   0.012
2023-09-01   0.018
2023-10-01   0.010
2023-11-01   0.030
2023-12-01   0.022
2024-01-01  -0.009
2024-02-01   0.033
2024-03-01   0.020
2024-04-01  -0.015
2024-05-01   0.005
2024-06-01   0.038
2024-07-01   0.028
2024-08-01   0.009
2024-09-01   0.007
2024-10-01   0.010
2024-11-01   0.009
2024-12-01  -0.018
2025-01-01   0.013
2025-02-01   0.025
2025-03-01  -0.019
2025-04-01   0.008
2025-05-01   0.012
2025-06-01   0.014
2025-07-01   0.002
2025-08-01   0.017
2025-09-01   0.041
2025-10-01   0.026
2025-11-01  -0.009
2025-12-01  

In [24]:
df_ts_clean.to_csv('./hf_returns/tidan.csv')

In [25]:
import pandas as pd

# Extracting data from the third image
# Columns: JAN, FEB, MAR, APR, MAY, JUN, JUL, AUG, SEP, OCT, NOV, DEC, YEAR
data_three = {
    2020: [None, None, -2.00, 2.72, 0.04, 1.92, -0.85, -1.27, 0.14, 0.18, 1.97, 1.04, 3.86],
    2021: [-0.42, -5.08, 1.68, 3.22, 6.30, 2.73, 3.15, 2.97, 5.21, 0.65, 2.91, 5.34, 32.08],
    2022: [5.73, 3.88, 0.63, 1.70, 0.45, 1.63, 2.99, 4.92, -5.26, 5.06, 0.26, 1.05, 25.05],
    2023: [1.77, 0.39, 0.61, 0.24, 0.64, 1.45, -0.57, -0.54, -0.61, -0.45, 1.34, -0.08, 4.23],
    2024: [1.51, -1.23, -0.08, 0.47, 0.80, 1.00, -0.54, -0.10, -1.65, 1.72, 1.30, 0.89, 4.12],
    2025: [0.98, 1.75, 2.66, 2.78, 1.37, -1.72, 1.13, 1.40, 0.08, -0.97, -0.71, 1.11, 10.21],
    2026: [0.36, 1.12, -6.15, None, None, None, None, None, None, None, None, None, -4.76]
}

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'YEAR']
df_three = pd.DataFrame(data_three, index=months)

# 1. Exclude the annual 'YEAR' aggregate row
df_months = df_three.loc[df_three.index != 'YEAR']

# 2. Reshape/Unstack from matrix format to long format
df_long = df_months.unstack().reset_index()
df_long.columns = ['Year', 'Month', 'Return']

# 3. Create numerical month mappings and construct full Datetime index
month_map = {m: i+1 for i, m in enumerate(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])}
df_long['Month_Num'] = df_long['Month'].map(month_map)
df_long['Date'] = pd.to_datetime(df_long['Year'].astype(str) + '-' + df_long['Month_Num'].astype(str) + '-01')

# Sort chronologically and keep relevant columns
df_ts = df_long.sort_values('Date').reset_index(drop=True)[['Date', 'Return']]

# 4. Filter out leading and trailing missing data to isolate the true timeline
first_valid = df_ts['Return'].first_valid_index()
last_valid = df_ts['Return'].last_valid_index()
df_ts_clean = df_ts.loc[first_valid:last_valid].copy().reset_index(drop=True)

# Set Date as the formal index for your time-series analysis
df_ts_clean.set_index('Date', inplace=True)

# Save to a new CSV file
df_ts_clean = df_ts_clean /100
# df_ts_clean.to_csv('monthly_returns_timeseries_3.csv')

print(df_ts_clean)

            Return
Date              
2020-03-01 -0.0200
2020-04-01  0.0272
2020-05-01  0.0004
2020-06-01  0.0192
2020-07-01 -0.0085
...            ...
2025-11-01 -0.0071
2025-12-01  0.0111
2026-01-01  0.0036
2026-02-01  0.0112
2026-03-01 -0.0615

[73 rows x 1 columns]


In [26]:
df_ts_clean.to_csv('./hf_returns/maple_cap.csv')

In [27]:
import pandas as pd

# Extracting data from the fourth image (AGSF LP Fund)
# Columns: Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec, YTD
data_four = {
    2013: [None, None, None, 1.13, 1.63, 2.13, 1.24, 1.79, 1.82, 2.38, 1.15, 2.79, 17.24],
    2014: [1.49, 2.28, 1.89, 1.12, 1.30, 1.08, -0.27, 2.28, 1.91, 0.13, 2.36, 2.67, 19.79],
    2015: [1.52, 1.43, 1.62, 1.31, 1.99, 1.86, 1.44, -8.79, 4.28, -8.97, 1.62, 2.00, 0.26],
    2016: [1.24, 1.58, 1.75, 1.55, 1.21, 1.85, 0.95, 0.84, 1.06, 0.83, 0.91, 0.76, 15.53],
    2017: [1.07, 1.05, 1.00, 0.99, 1.38, 0.83, 0.63, 1.09, 0.89, 1.24, 0.08, 1.03, 11.88],
    2018: [1.34, 3.17, 1.54, 1.28, 0.20, 0.64, 1.08, 1.18, 1.02, 1.14, 1.01, 1.22, 15.84],
    2019: [0.75, 0.76, 0.70, 0.78, 1.00, 0.32, 0.93, 0.57, 1.04, 0.98, 0.70, 1.14, 10.11],
    2020: [0.01, 1.19, 1.26, 1.15, 0.85, 0.47, 0.93, 0.82, 1.23, 1.07, -3.14, 0.10, 6.02],
    2021: [0.93, 0.61, 0.76, 0.43, 0.96, 0.67, 0.52, 0.62, 0.64, 0.63, 0.15, 1.26, 8.49],
    2022: [1.34, 0.63, 0.78, 0.53, 0.70, 0.69, 0.68, 0.84, 1.33, 0.29, -7.05, -6.39, -5.95],
    2023: [1.22, 0.42, 0.28, 0.68, 1.62, -0.15, 1.55, 0.89, 1.59, 2.15, 2.15, -0.08, 12.98],
    2024: [1.85, 1.48, -0.26, 0.75, 1.10, 1.45, 2.40, 1.09, 2.64, 3.32, 2.72, 1.22, 21.59],
    2025: [0.64, 2.67, 0.59, 1.30, 0.68, 0.30, 0.34, 0.52, 1.10, -0.59, 0.00, 1.72, 9.63],
    2026: [0.08, 0.25, 2.08, 1.16, None, None, None, None, None, None, None, None, 3.61]
}

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'YTD']
df_four = pd.DataFrame(data_four, index=months)

# 1. Exclude the annual 'YTD' aggregate row
df_months = df_four.loc[df_four.index != 'YTD']

# 2. Reshape/Unstack from matrix format to long format
df_long = df_months.unstack().reset_index()
df_long.columns = ['Year', 'Month', 'Return']

# 3. Create numerical month mappings and construct full Datetime index
month_map = {m: i+1 for i, m in enumerate(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])}
df_long['Month_Num'] = df_long['Month'].map(month_map)
df_long['Date'] = pd.to_datetime(df_long['Year'].astype(str) + '-' + df_long['Month_Num'].astype(str) + '-01')

# Sort chronologically and keep relevant columns
df_ts = df_long.sort_values('Date').reset_index(drop=True)[['Date', 'Return']]

# 4. Filter out leading and trailing missing data to isolate the true timeline
first_valid = df_ts['Return'].first_valid_index()
last_valid = df_ts['Return'].last_valid_index()
df_ts_clean = df_ts.loc[first_valid:last_valid].copy().reset_index(drop=True)

# Set Date as the formal index for your time-series analysis
df_ts_clean.set_index('Date', inplace=True)

# Save to a new CSV file
df_ts_clean = df_ts_clean /100

print(df_ts_clean)

            Return
Date              
2013-04-01  0.0113
2013-05-01  0.0163
2013-06-01  0.0213
2013-07-01  0.0124
2013-08-01  0.0179
...            ...
2025-12-01  0.0172
2026-01-01  0.0008
2026-02-01  0.0025
2026-03-01  0.0208
2026-04-01  0.0116

[157 rows x 1 columns]


In [28]:
df_ts_clean.to_csv('./hf_returns/global_sigma.csv')

In [29]:
import pandas as pd

# Extracting only the EADF-B (net) rows from the screenshot
# Columns: Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec, Year
data_five = {
    2016: [None, None, None, None, None, None, None, None, None, 1.23, -0.64, -0.14, 0.44],
    2017: [0.59, 0.84, 1.73, 2.62, 2.42, 1.19, 0.50, 1.41, 0.86, 0.93, 0.72, 1.37, 16.25],
    2018: [0.73, 0.83, 1.34, 1.50, -0.20, -0.21, 1.46, 0.82, -0.19, 0.60, 0.56, 1.01, 8.54],
    2019: [2.46, 0.97, 2.43, 2.54, 1.86, 1.58, 3.03, 1.49, 2.02, 1.60, 1.09, 1.55, 25.07],
    2020: [2.37, 2.85, 4.30, 3.31, 3.36, 2.05, 0.36, 1.19, -1.10, 1.77, 0.85, 1.53, 25.24],
    2021: [0.69, 1.25, -0.51, 0.37, 0.55, 1.07, 1.53, 2.32, 1.41, -0.06, 0.13, 0.32, 9.42],
    2022: [-0.92, -2.97, 0.25, -5.22, -1.90, -14.07, 2.72, 5.88, -3.57, 7.07, 12.78, 3.60, 1.03],
    2023: [0.26, -0.56, -3.38, -6.56, 10.33, 7.81, 3.38, -1.16, -0.72, 3.76, 1.50, 1.75, 16.37],
    2024: [-1.09, 3.66, 9.42, -1.43, 2.45, 1.88, -1.82, 2.16, 0.13, 3.58, 0.96, 5.00, 27.30],
    2025: [4.60, 3.33, 0.58, -0.42, 1.38, 2.78, 3.55, 1.29, 6.02, 3.77, 1.36, 2.84, 35.66],
    2026: [5.69, 4.26, -4.98, 1.49, None, None, None, None, None, None, None, None, 6.28]
}

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Year']
df_five = pd.DataFrame(data_five, index=months)

# 1. Exclude the annual 'Year' aggregate row
df_months = df_five.loc[df_five.index != 'Year']

# 2. Reshape/Unstack from matrix format to long format
df_long = df_months.unstack().reset_index()
df_long.columns = ['Year', 'Month', 'Return']

# 3. Create numerical month mappings and construct full Datetime index
month_map = {m: i+1 for i, m in enumerate(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])}
df_long['Month_Num'] = df_long['Month'].map(month_map)
df_long['Date'] = pd.to_datetime(df_long['Year'].astype(str) + '-' + df_long['Month_Num'].astype(str) + '-01')

# Sort chronologically and keep relevant columns
df_ts = df_long.sort_values('Date').reset_index(drop=True)[['Date', 'Return']]

# 4. Filter out leading and trailing missing data to isolate the true timeline
first_valid = df_ts['Return'].first_valid_index()
last_valid = df_ts['Return'].last_valid_index()
df_ts_clean = df_ts.loc[first_valid:last_valid].copy().reset_index(drop=True)

# Set Date as the formal index for your time-series analysis
df_ts_clean.set_index('Date', inplace=True)

# Save to a new CSV file
# df_ts_clean.to_csv('eadf_b_returns_timeseries.csv')
df_ts_clean = df_ts_clean /100

print(df_ts_clean)

            Return
Date              
2016-10-01  0.0123
2016-11-01 -0.0064
2016-12-01 -0.0014
2017-01-01  0.0059
2017-02-01  0.0084
...            ...
2025-12-01  0.0284
2026-01-01  0.0569
2026-02-01  0.0426
2026-03-01 -0.0498
2026-04-01  0.0149

[115 rows x 1 columns]


In [30]:
df_ts_clean.to_csv('./hf_returns/enko.csv')

In [3]:
import io
import pandas as pd

# Cleaned data string
csv_data = """Month,Return
2008-01-31,1.83
2008-02-29,7.54
2008-03-31,-4.65
2008-04-30,5.95
2008-05-31,5.09
2008-06-30,4.91
2008-07-31,-2.87
2008-08-31,-0.43
2008-09-30,1.16
2008-10-31,1.20
2008-11-30,9.40
2008-12-31,3.69
2009-01-31,4.49
2009-02-28,4.09
2009-03-31,3.01
2009-04-30,0.76
2009-05-31,12.18
2009-06-30,-0.24
2009-07-31,3.68
2009-08-31,3.85
2009-09-30,4.51
2009-10-31,0.04
2009-11-30,8.66
2009-12-31,2.95
2010-01-31,1.66
2010-02-28,2.85
2010-03-31,5.34
2010-04-30,0.18
2010-05-31,3.47
2010-06-30,4.02
2010-07-31,5.45
2010-08-31,6.78
2010-09-30,4.87
2010-10-31,4.89
2010-11-30,0.57
2010-12-31,8.42
2011-01-31,10.49
2011-02-28,1.28
2011-03-31,2.61
2011-04-30,2.82
2011-10-31,3.92
2011-11-30,7.37
2011-12-31,2.56
2012-01-31,4.07
2012-02-29,2.02
2012-03-31,1.65
2012-04-30,1.34
2012-05-31,0.30
2012-06-30,-3.89
2012-07-31,5.73
2012-08-31,2.41
2012-09-30,-0.42
2012-10-31,0.53
2012-11-30,-1.23
2012-12-31,0.57
2013-01-31,2.99
2013-02-28,1.13
2013-03-31,0.48
2013-04-30,-2.33
2013-05-31,-0.32
2013-06-30,0.73
2013-07-31,0.45
2013-08-31,-5.05
2013-09-30,1.79
2013-10-31,0.30
2013-11-30,2.16
2013-12-31,3.70
2014-01-31,3.73
2014-02-28,-4.69
2014-03-31,-2.28
2014-04-30,3.35
2014-05-31,0.97
2014-06-30,0.01
2014-07-31,2.03
2014-08-31,4.09
2014-09-30,-0.02
2014-10-31,-3.10
2014-11-30,-0.12
2014-12-31,3.42
2015-01-31,1.11
2015-02-28,0.50
2015-03-31,0.65
2015-04-30,1.72
2015-05-31,2.52
2015-06-30,-0.19
2015-07-31,-1.67
2015-08-31,-3.86
2015-09-30,1.57
2015-10-31,-0.07
2015-11-30,0.86
2015-12-31,6.43
2016-01-31,3.16
2016-02-29,3.03
2016-03-31,2.93
2016-04-30,-4.11
2016-05-31,3.11
2016-06-30,-3.24
2016-07-31,-1.14
2016-08-31,3.34
2016-09-30,-1.09
2016-10-31,4.06
2016-11-30,1.23
2016-12-31,1.41
2017-01-31,2.78
2017-02-28,2.57
2017-03-31,1.20
2017-04-30,-0.49
2017-05-31,-1.14
2017-06-30,1.21
2017-07-31,-0.07
2017-08-31,1.79
2017-09-30,-0.58
2017-10-31,3.18
2017-11-30,4.54
2017-12-31,1.71
2018-01-31,2.60
2018-02-28,1.18
2018-03-31,-1.06
2018-04-30,2.11
2018-05-31,0.97
2018-06-30,-1.60
2018-07-31,5.77
2018-08-31,0.37
2018-09-30,-0.94
2018-10-31,-0.19
2018-11-30,0.32
2018-12-31,8.66
2019-01-31,2.10
2019-02-28,3.21
2019-03-31,2.36
2019-04-30,4.67
2019-05-31,-4.12
2019-06-30,2.21
2019-07-31,2.14
2019-08-31,-0.23
2019-09-30,-0.90
2019-10-31,5.89
2019-11-30,3.17
2019-12-31,3.74
2020-01-31,-1.12
2020-02-29,-3.89
2020-03-31,-8.06
2020-04-30,9.29
2020-05-31,6.77
2020-06-30,5.19
2020-07-31,2.13
2020-08-31,0.26
2020-09-30,0.55
2020-10-31,9.51
2020-11-30,4.67
2020-12-31,5.21
2021-01-31,4.22
2021-02-28,0.74
2021-03-31,3.25
2021-04-30,5.76
2021-05-31,3.22
2021-06-30,5.39
2021-07-31,1.89
2021-08-31,7.40
2021-09-30,2.32
2021-10-31,3.29
2021-11-30,4.12
2021-12-31,6.21
2022-01-31,6.12
2022-02-28,-0.12
2022-03-31,1.58
2022-04-30,7.76
2022-05-31,8.06
2022-06-30,-0.37
2022-07-31,0.30
2022-08-31,5.58
2022-09-30,2.00
2022-10-31,6.19
2022-11-30,3.72
2022-12-31,3.65
2023-01-31,5.52
2023-02-28,6.69
2023-03-31,0.06
2023-04-30,1.49
2023-05-31,5.36
2023-06-30,0.22
2023-07-31,3.55
2023-08-31,6.06
2023-09-30,-0.59
2023-10-31,4.02
2023-11-30,2.04
2023-12-31,1.37
2024-01-31,7.68
2024-02-29,4.64
2024-03-31,1.65
2024-04-30,-2.03
2024-05-31,3.82
2024-06-30,3.03
2024-07-31,1.04
2024-08-31,1.82
2024-09-30,-6.59
2024-10-31,9.39
2024-11-30,7.63
2024-12-31,5.76
2025-01-31,-1.58
2025-02-28,2.58
2025-03-31,5.04
2025-04-30,-10.51
2025-05-31,6.72
2025-06-30,3.51
2025-07-31,4.65
2025-08-31,5.48
2025-09-30,2.67
2025-10-31,3.55
2025-11-30,4.22
2025-12-31,1.43
2026-01-31,-4.94
2026-02-27,4.55
2026-03-31,-2.82
2026-04-30,8.34"""

# Parse the data directly into a DataFrame
df = pd.read_csv(io.StringIO(csv_data))

# Convert Month column to datetime objects
df["Month"] = pd.to_datetime(df["Month"])

# Apply your net return formula: (Value * 0.9) - (1% / 12)
df["Net_Return"] = (df["Return"] * 0.9) - (0.01 / 12)

print(df.head())

       Month  Return  Net_Return
0 2008-01-31    1.83    1.646167
1 2008-02-29    7.54    6.785167
2 2008-03-31   -4.65   -4.185833
3 2008-04-30    5.95    5.354167
4 2008-05-31    5.09    4.580167


In [8]:
(df.set_index('Month')/100)['Net_Return'].to_csv('./hf_returns/quanstream.csv')

In [9]:
import yfinance as yf

# Download historical data
tickers = ['^SP500TR']
data = yf.download(tickers, start='2000-01-01', interval='1mo')

sp500 = data['Close']

sp500.columns = ['SP500TR']

returns = sp500.pct_change().dropna()

returns

/var/folders/mc/qf75k40s6ns_nr8c35wdmp400000gn/T/ipykernel_13094/1224205126.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start='2000-01-01', interval='1mo')
[*********************100%***********************]  1 of 1 completed


,SP500TR
Date,
2000-02-01,-0.018929
2000-03-01,0.097829
2000-04-01,-0.030086
2000-05-01,-0.020518
2000-06-01,0.024654
...,...
2026-01-01,0.014500
2026-02-01,-0.007600
2026-03-01,-0.049795


In [11]:
returns.to_csv('./hf_returns/sp500.csv')